# MedFrameQA 顺序投票 + 条件 top-2 纠错

这份 notebook 使用统一论文协议：

- 先确认模型服务和 frozen manifest；
- 再做 1-sample sanity 和 5-sample smoke；
- 正式实验直接在当前 cell 内阻塞运行，不再依赖 `tmux`；
- 跑完后直接本地画出 generation 曲线；
- 最后导出 valid/invalid rate、top-3、holdout、final test 的 paper summary。


In [ ]:
!nvidia-smi


`8000/8001/8002` 模型服务建议通过 `/gluon4/xl693/start_models.sh` 在 `tmux` 里常驻。  
本 notebook 不会去停止模型服务；如果要中断实验，直接中断当前运行 cell 即可。


In [ ]:
import os
import time

# 这格只负责声明本 notebook 的运行预算与公共环境变量。
TASK_DIR = os.path.abspath("advanced_vqa_task_order_vote_plus")
TASK_NAME = "advanced_vqa_task_order_vote_plus"
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
RESULTS_DIR = os.path.abspath(os.path.join("results", f"advanced_vqa_task_order_vote_plus_{RUN_TAG}"))
SESSION_NAME = f"advanced_vqa_task_order_vote_plus_{RUN_TAG}"
SPLIT_MANIFEST_PATH = os.path.abspath("medframeqa_split_manifest_v1.json")
NUM_GENERATIONS = 50
SEARCH_MINI_SIZE = 256
POOL_REEVAL_GENS = [10, 20, 30, 40, 49]
INVALID_RATE_THRESHOLD = 0.05
MAX_PROPOSAL_JOBS = 4
MAX_EVALUATION_JOBS = 1
DEFAULT_ENV = {
    "MEDFRAMEQA_SPLIT_MANIFEST": SPLIT_MANIFEST_PATH,
    "MEDFRAMEQA_PROTOCOL_MODE": "search_mini",
    "MEDFRAMEQA_SEARCH_MINI_SIZE": "256",
    "MEDFRAMEQA_POOL_REEVAL_GENS": "10,20,30,40,49",
    "MEDFRAMEQA_API_TIMEOUT": "30",
    "MEDFRAMEQA_REQUEST_RETRIES": "3",
    "MEDFRAMEQA_REQUEST_RETRY_SLEEP": "2.0",
    "MEDFRAMEQA_VLM_LOCK_PATH": "/tmp/medframeqa_vlm_8001.lock",
    "MEDFRAMEQA_VLM_LOCK_POLL_SECONDS": "5",
    "MEDFRAMEQA_VLM_LOCK_STALE_SECONDS": "1800",
    "MEDFRAMEQA_IMAGE_MAX_SIDE": "384",
    "MEDFRAMEQA_IMAGE_QUALITY": "60",
}

for key, value in DEFAULT_ENV.items():
    os.environ[key] = value

print("TASK_DIR:", TASK_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("SESSION_NAME:", SESSION_NAME)
print("SPLIT_MANIFEST_PATH:", SPLIT_MANIFEST_PATH)
print("NUM_GENERATIONS:", NUM_GENERATIONS)
print("SEARCH_MINI_SIZE:", SEARCH_MINI_SIZE)
print("POOL_REEVAL_GENS:", POOL_REEVAL_GENS)
print("INVALID_RATE_THRESHOLD:", INVALID_RATE_THRESHOLD)
print("DEFAULT_ENV:", DEFAULT_ENV)


In [ ]:
# 检查 8000 / 8001 / 8002 三个服务是否可用。
from openai import OpenAI

endpoints = {
    "coder": "http://localhost:8000/v1",
    "vlm": "http://localhost:8001/v1",
    "embed": "http://localhost:8002/v1",
}

for name, base_url in endpoints.items():
    try:
        client = OpenAI(api_key="local", base_url=base_url, timeout=10)
        models = client.models.list()
        model_ids = [model.id for model in models.data[:5]]
        print(name, "OK", model_ids)
    except Exception as exc:
        print(name, "ERROR", exc)


In [ ]:
%%writefile medframeqa_runtime.py
"""
MedFrameQA 共享运行时工具。

这个文件负责 5 类共用能力：
1. 数据集与 frozen split manifest 的加载。
2. 多图样本的图像压缩、消息构造、选项字母解析。
3. 单一 8001 VLM 服务下的全局串行锁，避免多个实验同时打同一个服务。
4. 统一的实验指标字段、结果汇总、异常兜底。
5. notebook 与 evaluate.py 共用的小工具，尽量把脆弱逻辑收口到这里。

输入：
- MedFrameQA 数据集
- 每条任务线生成的 initial.py / evaluate.py
- notebook 的 sanity / smoke / paper eval 请求

输出：
- 规范化后的模型消息
- 统一 schema 的 metrics
- VLM 锁状态与运行控制信息

失败处理：
- 如果变异程序返回了非法配置，会尽量回退到默认值而不是直接崩。
- 如果请求 8001 失败，会按统一重试策略重试。
- 如果实验内部抛异常，会转成 invalid_generation 指标，而不是让整代静默失败。
"""

import base64
import csv
import hashlib
import importlib.util
import io
import json
import math
import os
import re
import signal
import string
import sys
import time
import traceback
import uuid
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path

from PIL import Image
from datasets import load_dataset
from openai import OpenAI


DEFAULT_PROJECT_ROOT = Path("/gluon4/xl693/evolve")
DEFAULT_DATASET_ID = os.environ.get("MEDFRAMEQA_DATASET", "SuhaoYu1020/MedFrameQA")
DEFAULT_DATASET_CACHE_DIR = os.environ.get(
    "MEDFRAMEQA_DATASET_CACHE",
    "/tmp/medframeqa_hf_cache",
)
DEFAULT_SPLIT_MANIFEST = os.environ.get(
    "MEDFRAMEQA_SPLIT_MANIFEST",
    str(DEFAULT_PROJECT_ROOT / "medframeqa_split_manifest_v1.json"),
)
DEFAULT_IMAGE_MAX_SIDE = int(os.environ.get("MEDFRAMEQA_IMAGE_MAX_SIDE", "384"))
DEFAULT_IMAGE_QUALITY = int(os.environ.get("MEDFRAMEQA_IMAGE_QUALITY", "60"))
DEFAULT_MAX_IMAGES = int(os.environ.get("MEDFRAMEQA_MAX_IMAGES", "0"))
DEFAULT_API_TIMEOUT = float(os.environ.get("MEDFRAMEQA_API_TIMEOUT", "30"))
DEFAULT_REQUEST_RETRIES = int(os.environ.get("MEDFRAMEQA_REQUEST_RETRIES", "3"))
DEFAULT_REQUEST_RETRY_SLEEP = float(
    os.environ.get("MEDFRAMEQA_REQUEST_RETRY_SLEEP", "2.0")
)
DEFAULT_VLM_LOCK_PATH = Path(
    os.environ.get("MEDFRAMEQA_VLM_LOCK_PATH", "/tmp/medframeqa_vlm_8001.lock")
)
DEFAULT_VLM_LOCK_POLL_SECONDS = float(
    os.environ.get("MEDFRAMEQA_VLM_LOCK_POLL_SECONDS", "5")
)
DEFAULT_VLM_LOCK_STALE_SECONDS = float(
    os.environ.get("MEDFRAMEQA_VLM_LOCK_STALE_SECONDS", "1800")
)
RESAMPLING = Image.Resampling.LANCZOS if hasattr(Image, "Resampling") else Image.LANCZOS
IMAGE_FRAME_PATTERN = re.compile(r"^image_(\d+)$")

PROTOCOL_ALIAS = {
    "search_mini": "search",
    "selection_holdout": "holdout",
    "independent_final_test": "final_test",
    "evolution_pool": "evolution_pool",
}


def stable_hash(text):
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16)


def resolve_project_root(start_path=None):
    if start_path is None:
        path = Path.cwd()
    else:
        path = Path(start_path).resolve()

    candidates = [path] if path.is_dir() else [path.parent]
    candidates.extend(path.parents)
    for candidate in candidates:
        if (candidate / "medframeqa_runtime.py").exists():
            return candidate
        if (candidate / "build_medframeqa_notebooks.py").exists():
            return candidate
    return DEFAULT_PROJECT_ROOT


def ensure_project_root_on_path(start_path=None):
    root = resolve_project_root(start_path)
    root_str = str(root)
    if root_str not in sys.path:
        sys.path.insert(0, root_str)
    return root


def load_mutated_module(path):
    spec = importlib.util.spec_from_file_location("mutated_program", path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def merge_text_config(defaults, candidate):
    """只把白名单里的字符串/嵌套字符串字段从 candidate 投影回默认配置。"""
    merged = json.loads(json.dumps(defaults))
    if not isinstance(candidate, dict):
        return merged

    for key, fallback in defaults.items():
        value = candidate.get(key)
        if isinstance(fallback, dict):
            if isinstance(value, dict):
                for subkey, subfallback in fallback.items():
                    subvalue = value.get(subkey)
                    if isinstance(subfallback, str):
                        if isinstance(subvalue, str) and subvalue:
                            merged[key][subkey] = subvalue
                    else:
                        merged[key][subkey] = subfallback
        elif isinstance(fallback, str):
            if isinstance(value, str) and value:
                merged[key] = value
    return merged


def parse_json_config_block(raw_text):
    """
    把可进化 JSON 文本解析成 dict。
    解析失败时返回空 dict，这样上层会自动回退到默认配置。
    """
    if not isinstance(raw_text, str) or not raw_text.strip():
        return {}
    try:
        value = json.loads(raw_text)
    except Exception:
        return {}
    return value if isinstance(value, dict) else {}


def coerce_choice(value, allowed_values, default_value):
    return value if value in allowed_values else default_value


def coerce_int_choice(value, allowed_values, default_value):
    return value if value in allowed_values else default_value


def prepare_image(image):
    rgb = image.convert("RGB")
    max_side = max(
        128,
        int(os.environ.get("MEDFRAMEQA_IMAGE_MAX_SIDE", str(DEFAULT_IMAGE_MAX_SIDE))),
    )
    quality = max(
        20,
        min(
            95,
            int(
                os.environ.get(
                    "MEDFRAMEQA_IMAGE_QUALITY",
                    str(DEFAULT_IMAGE_QUALITY),
                )
            ),
        ),
    )

    if max(rgb.size) > max_side:
        scale = max_side / max(rgb.size)
        new_size = (
            max(1, int(round(rgb.size[0] * scale))),
            max(1, int(round(rgb.size[1] * scale))),
        )
        rgb = rgb.resize(new_size, RESAMPLING)
    return rgb, quality


def encode_image(image):
    if not isinstance(image, Image.Image):
        return None
    rgb, quality = prepare_image(image)
    buffered = io.BytesIO()
    rgb.save(buffered, format="JPEG", quality=quality, optimize=True)
    return base64.b64encode(buffered.getvalue()).decode("utf-8")


def is_frame_image_column(column_name):
    """
    只把真正的逐帧图像列当成图片输入。

    MedFrameQA 的样本里同时存在:
    - image_url: 原始 URL / 辅助字符串
    - image_1 ... image_5: 真正的图像帧

    这里必须显式排除 image_url，否则会把题目的图像数错误地整体 +1，
    进而污染 prompt 元信息、图像编号和按 frame count 的统计分析。
    """
    return bool(IMAGE_FRAME_PATTERN.match(str(column_name)))


def image_column_sort_key(column_name):
    match = IMAGE_FRAME_PATTERN.match(str(column_name))
    return int(match.group(1)) if match else math.inf


def get_image_columns(sample):
    image_columns = [
        key
        for key in sample.keys()
        if is_frame_image_column(key) and sample[key]
    ]
    image_columns.sort(key=image_column_sort_key)
    max_images = int(os.environ.get("MEDFRAMEQA_MAX_IMAGES", str(DEFAULT_MAX_IMAGES)))
    if max_images > 0:
        image_columns = image_columns[:max_images]
    return image_columns


def format_options(options, option_indices=None, letters=None):
    if option_indices is None:
        option_indices = list(range(len(options)))
    labels = letters or string.ascii_uppercase
    return "\n".join(
        f"{labels[local_idx]}. {options[option_idx]}"
        for local_idx, option_idx in enumerate(option_indices)
    )


def infer_case_metadata(sample):
    combined_text = " ".join(
        [
            sample.get("system", ""),
            sample.get("organ", ""),
            sample.get("keyword", ""),
            sample.get("modality", ""),
            sample.get("question", ""),
            *sample.get("options", []),
        ]
    ).lower()
    image_columns = get_image_columns(sample)

    sequence_tokens = [
        "before",
        "after",
        "follow-up",
        "progression",
        "sequence",
        "change",
        "stable",
        "improved",
        "worse",
        "postoperative",
    ]

    present_sequence_tokens = [token for token in sequence_tokens if token in combined_text]
    return {
        "image_count": len(image_columns),
        "system": sample.get("system", "unspecified"),
        "organ": sample.get("organ", "unspecified"),
        "keyword": sample.get("keyword", "unspecified"),
        "modality": sample.get("modality", "unspecified"),
        "video_id": sample.get("video_id", "unknown"),
        "sequence_hints": present_sequence_tokens[:5] or ["none"],
    }


def render_metadata_block(metadata):
    return (
        "Structured case metadata:\n"
        f"- System: {metadata['system']}\n"
        f"- Organ: {metadata['organ']}\n"
        f"- Keyword: {metadata['keyword']}\n"
        f"- Modality: {metadata['modality']}\n"
        f"- Video ID: {metadata['video_id']}\n"
        f"- Ordered image count: {metadata['image_count']}\n"
        f"- Sequence hints: {', '.join(metadata['sequence_hints'])}\n"
    )


def build_content_list(sample, prompt_text, image_prompt_template):
    content = [{"type": "text", "text": prompt_text}]
    image_columns = get_image_columns(sample)
    for index, column in enumerate(image_columns, 1):
        encoded = encode_image(sample[column])
        if encoded:
            if image_prompt_template:
                content.append(
                    {
                        "type": "text",
                        "text": "\n" + image_prompt_template.format(index=index, total=len(image_columns)),
                    }
                )
            content.append(
                {
                    "type": "image_url",
                    "image_url": {"url": f"data:image/jpeg;base64,{encoded}"},
                }
            )
    return content, image_columns


def get_manifest_path(manifest_path=None):
    if manifest_path:
        return Path(manifest_path)
    return Path(os.environ.get("MEDFRAMEQA_SPLIT_MANIFEST", DEFAULT_SPLIT_MANIFEST))


def write_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")


def write_csv_rows(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    rows = list(rows)
    if not rows:
        path.write_text("")
        return
    fieldnames = sorted({key for row in rows for key in row.keys()})
    with path.open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow(row)


def load_medframeqa_dataset(include_images=True):
    dataset = load_dataset(
        DEFAULT_DATASET_ID,
        split="test",
        cache_dir=DEFAULT_DATASET_CACHE_DIR,
    )
    if not include_images:
        drop_columns = [column for column in dataset.column_names if column.startswith("image_")]
        dataset = dataset.remove_columns(drop_columns)
    return dataset


def load_split_manifest(manifest_path=None):
    path = get_manifest_path(manifest_path)
    if not path.exists():
        ensure_split_manifest(path)
    else:
        ensure_project_root_on_path(__file__)
        from create_medframeqa_split_manifest import GENERATOR_STRATEGY, MANIFEST_VERSION

        try:
            payload = json.loads(path.read_text())
        except Exception:
            ensure_split_manifest(path)
        else:
            generator = payload.get("generator", {})
            if payload.get("version") != MANIFEST_VERSION or generator.get("strategy") != GENERATOR_STRATEGY:
                ensure_split_manifest(path)
    return json.loads(path.read_text())


def ensure_split_manifest(manifest_path=None):
    path = get_manifest_path(manifest_path)
    ensure_project_root_on_path(__file__)
    from create_medframeqa_split_manifest import (
        GENERATOR_STRATEGY,
        MANIFEST_VERSION,
        ensure_split_manifest as _ensure,
    )

    if path.exists():
        try:
            payload = json.loads(path.read_text())
        except Exception:
            payload = None
        if payload:
            generator = payload.get("generator", {})
            if payload.get("version") == MANIFEST_VERSION and generator.get("strategy") == GENERATOR_STRATEGY:
                return path

    return Path(_ensure(output_path=path))


def build_question_index(dataset):
    return {question_id: index for index, question_id in enumerate(dataset["question_id"])}


def select_stratified_search_mini(question_ids, meta_by_qid, size):
    if size <= 0 or size >= len(question_ids):
        return list(question_ids)

    strata = defaultdict(list)
    for question_id in question_ids:
        answer, modality = meta_by_qid[question_id]
        strata[(answer, modality)].append(question_id)

    sorted_items = sorted(strata.items(), key=lambda item: stable_hash(f"{item[0][0]}::{item[0][1]}"))
    quotas = {}
    remainders = []
    allocated = 0
    total = len(question_ids)

    for key, members in sorted_items:
        members.sort(key=stable_hash)
        ideal = size * len(members) / total
        quota = min(len(members), int(math.floor(ideal)))
        quotas[key] = quota
        allocated += quota
        remainders.append((ideal - quota, stable_hash(str(key)), key))

    remaining = size - allocated
    for _, _, key in sorted(remainders, key=lambda item: (-item[0], item[1])):
        if remaining <= 0:
            break
        if quotas[key] < len(strata[key]):
            quotas[key] += 1
            remaining -= 1

    if remaining > 0:
        leftovers = []
        for key, members in sorted_items:
            leftovers.extend(members[quotas[key] :])
        for question_id in sorted(leftovers, key=stable_hash)[:remaining]:
            answer, modality = meta_by_qid[question_id]
            quotas[(answer, modality)] += 1

    selected = []
    for key, members in sorted_items:
        selected.extend(members[: quotas[key]])
    return sorted(selected, key=stable_hash)


def get_protocol_subset(dataset, manifest, protocol_mode, search_mini_size):
    question_index = build_question_index(dataset)
    meta_by_qid = {
        qid: (answer, modality)
        for qid, answer, modality in zip(
            dataset["question_id"],
            dataset["correct_answer"],
            dataset["modality"],
        )
    }

    if protocol_mode == "search_mini":
        pool_ids = manifest["splits"]["evolution_pool"]
        selected_ids = select_stratified_search_mini(pool_ids, meta_by_qid, search_mini_size)
    else:
        selected_ids = manifest["splits"][protocol_mode]

    indices = [question_index[question_id] for question_id in selected_ids]
    subset = dataset.select(indices)
    return subset, selected_ids


def get_protocol_meta(manifest, protocol_mode, selected_ids, search_mini_size):
    return {
        "protocol_mode": protocol_mode,
        "split_manifest_version": manifest["version"],
        "selected_subset_size": len(selected_ids),
        "search_mini_size": search_mini_size,
        "selected_subset_preview": selected_ids[:5],
    }


def protocol_alias(protocol_mode):
    return PROTOCOL_ALIAS.get(protocol_mode, protocol_mode)


def protocol_metric_prefix(protocol_mode):
    return protocol_alias(protocol_mode)


def make_protocol_metrics(protocol_mode, correct, total):
    prefix = protocol_metric_prefix(protocol_mode)
    return {
        f"{prefix}_score": correct / total if total else 0.0,
        f"{prefix}_correct": correct,
        f"{prefix}_size": total,
    }


def get_protocol_score_key(protocol_mode):
    return f"{protocol_metric_prefix(protocol_mode)}_score"


def get_protocol_size_key(protocol_mode):
    return f"{protocol_metric_prefix(protocol_mode)}_size"


def normalize_messages(payload):
    if not isinstance(payload, list) or not payload:
        raise TypeError("format_vqa_payload must return a non-empty list")
    if isinstance(payload[0], dict) and "role" in payload[0]:
        return payload
    return [{"role": "user", "content": payload}]


def extract_option_letter(text, valid_letters):
    raw = (text or "").strip().upper()
    if raw in valid_letters:
        return raw
    for letter in valid_letters:
        if re.search(rf"\b{re.escape(letter)}\b", raw):
            return letter
    if raw and raw[0] in valid_letters:
        return raw[0]
    return ""


def get_image_budget_schedule():
    schedule = [
        (DEFAULT_IMAGE_MAX_SIDE, DEFAULT_IMAGE_QUALITY),
        (320, 50),
        (256, 40),
    ]
    ordered = []
    seen = set()
    for item in schedule:
        if item not in seen:
            ordered.append(item)
            seen.add(item)
    return ordered


def is_context_length_error(exc):
    message = str(exc).lower()
    return "maximum context length" in message or "input length" in message


def set_image_budget(max_side, quality):
    os.environ["MEDFRAMEQA_IMAGE_MAX_SIDE"] = str(max_side)
    os.environ["MEDFRAMEQA_IMAGE_QUALITY"] = str(quality)


def is_retryable_request_error(exc):
    name = type(exc).__name__.lower()
    message = str(exc).lower()
    retryable_names = (
        "apiconnectionerror",
        "apitimeouterror",
        "connecterror",
        "connecttimeout",
        "readtimeout",
        "pooltimeout",
        "remoteprotocolerror",
        "serverdisconnectederror",
        "readerror",
    )
    retryable_message_parts = (
        "connection error",
        "connection refused",
        "connection reset",
        "server disconnected",
        "temporarily unavailable",
        "timed out",
        "timeout",
        "502",
        "503",
        "504",
    )
    return any(token in name for token in retryable_names) or any(
        token in message for token in retryable_message_parts
    )


def make_openai_client(base_url, timeout=None):
    return OpenAI(
        api_key="local",
        base_url=base_url,
        timeout=timeout if timeout is not None else DEFAULT_API_TIMEOUT,
    )


def _lock_meta_path(lock_path):
    return Path(str(lock_path) + ".json")


def _pid_exists(pid):
    if pid is None:
        return False
    try:
        os.kill(int(pid), 0)
    except (OSError, ValueError):
        return False
    return True


def read_vlm_lock_info(lock_path=None):
    lock_path = Path(lock_path or DEFAULT_VLM_LOCK_PATH)
    meta_path = _lock_meta_path(lock_path)
    if not meta_path.exists():
        return {}
    try:
        return json.loads(meta_path.read_text())
    except Exception:
        return {}


def _write_vlm_lock_info(lock_path, info):
    meta_path = _lock_meta_path(lock_path)
    meta_path.write_text(json.dumps(info, indent=2, ensure_ascii=False) + "\n")


def _remove_vlm_lock_files(lock_path):
    lock_path = Path(lock_path)
    meta_path = _lock_meta_path(lock_path)
    try:
        lock_path.unlink()
    except FileNotFoundError:
        pass
    try:
        meta_path.unlink()
    except FileNotFoundError:
        pass


def _is_vlm_lock_stale(lock_path, stale_seconds):
    lock_path = Path(lock_path)
    if not lock_path.exists():
        return False
    info = read_vlm_lock_info(lock_path)
    pid = info.get("pid")
    if pid is not None and not _pid_exists(pid):
        return True
    try:
        age = time.time() - lock_path.stat().st_mtime
    except FileNotFoundError:
        return False
    if age > stale_seconds and pid is not None and not _pid_exists(pid):
        return True
    return False


@contextmanager
def acquire_vlm_lock(task_name, results_dir=None, mode="eval", lock_path=None):
    """
    在单个 8001 VLM 服务前面加一把全局文件锁。

    设计目标：
    - notebook / smoke / evaluate / paper eval 全部共用同一把锁；
    - 后来的实验不假死，而是明确显示自己在等待谁；
    - 旧 owner 已死时，锁可以自动回收。
    """
    lock_path = Path(lock_path or DEFAULT_VLM_LOCK_PATH)
    lock_path.parent.mkdir(parents=True, exist_ok=True)
    poll_seconds = DEFAULT_VLM_LOCK_POLL_SECONDS
    stale_seconds = DEFAULT_VLM_LOCK_STALE_SECONDS
    token = uuid.uuid4().hex
    info = {
        "token": token,
        "pid": os.getpid(),
        "task_name": task_name,
        "results_dir": str(results_dir) if results_dir else "",
        "mode": mode,
        "started_at": time.time(),
    }
    wait_started = time.time()
    last_report = 0.0

    while True:
        try:
            fd = os.open(str(lock_path), os.O_CREAT | os.O_EXCL | os.O_WRONLY)
            with os.fdopen(fd, "w") as handle:
                handle.write(token + "\n")
            _write_vlm_lock_info(lock_path, info)
            break
        except FileExistsError:
            if _is_vlm_lock_stale(lock_path, stale_seconds):
                print(f"[VLM LOCK] 清理陈旧锁: {lock_path}", file=sys.stderr)
                _remove_vlm_lock_files(lock_path)
                continue

            owner = read_vlm_lock_info(lock_path)
            now = time.time()
            if now - last_report >= poll_seconds:
                owner_task = owner.get("task_name", "unknown")
                owner_mode = owner.get("mode", "unknown")
                owner_pid = owner.get("pid", "unknown")
                owner_dir = owner.get("results_dir", "")
                print(
                    f"[VLM LOCK] 等待 8001: owner_task={owner_task} owner_mode={owner_mode} "
                    f"owner_pid={owner_pid} owner_results={owner_dir}",
                    file=sys.stderr,
                )
                last_report = now
            time.sleep(poll_seconds)

    wait_seconds = round(time.time() - wait_started, 3)
    info["waited_for_lock_sec"] = wait_seconds
    _write_vlm_lock_info(lock_path, info)
    try:
        yield info
    finally:
        current = read_vlm_lock_info(lock_path)
        if current.get("token") == token:
            _remove_vlm_lock_files(lock_path)


def generate_guided_choice(
    client,
    messages,
    valid_choices,
    model,
    call_stats=None,
):
    if call_stats is not None:
        call_stats["vlm_call_count"] = call_stats.get("vlm_call_count", 0) + 1

    extra_body = {"guided_choice": valid_choices}
    if messages[-1]["role"] == "assistant":
        extra_body["continue_final_message"] = True
        extra_body["add_generation_prompt"] = False

    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=0.0,
        max_tokens=4,
        extra_body=extra_body,
    )
    return response.choices[0].message.content or ""


def generate_guided_choice_with_retries(
    client,
    message_builder,
    valid_choices,
    model,
    debug_label,
    call_stats=None,
    request_retries=None,
    request_retry_sleep=None,
):
    request_retries = (
        DEFAULT_REQUEST_RETRIES if request_retries is None else request_retries
    )
    request_retry_sleep = (
        DEFAULT_REQUEST_RETRY_SLEEP
        if request_retry_sleep is None
        else request_retry_sleep
    )
    last_exc = None

    for max_side, quality in get_image_budget_schedule():
        set_image_budget(max_side, quality)
        attempt = 0
        while True:
            try:
                messages = normalize_messages(message_builder())
                return generate_guided_choice(
                    client,
                    messages,
                    valid_choices,
                    model,
                    call_stats=call_stats,
                )
            except Exception as exc:
                last_exc = exc
                if is_context_length_error(exc):
                    print(
                        f"{debug_label} retry image_max_side={max_side} "
                        f"image_quality={quality} error={exc}",
                        file=sys.stderr,
                    )
                    break
                if is_retryable_request_error(exc) and attempt < request_retries:
                    attempt += 1
                    sleep_s = request_retry_sleep * attempt
                    print(
                        f"{debug_label} request_retry={attempt}/{request_retries} "
                        f"sleep={sleep_s:.1f}s error={exc}",
                        file=sys.stderr,
                    )
                    time.sleep(sleep_s)
                    continue
                raise
    raise last_exc


def get_option_orders(num_options, order_views):
    base = list(range(num_options))
    candidates = [
        ("identity", base),
        ("reverse", list(reversed(base))),
        ("rotate_left", base[1:] + base[:1] if num_options > 1 else base),
        ("outside_in", base[::2] + base[1::2] if num_options > 2 else base),
    ]
    orders = []
    seen = set()
    for order_name, order in candidates:
        key = tuple(order)
        if key not in seen:
            orders.append((order_name, order))
            seen.add(key)
        if len(orders) >= order_views:
            break
    return orders


def local_to_global(local_choice, option_indices):
    if not isinstance(local_choice, str) or len(local_choice) != 1:
        return ""
    if local_choice not in string.ascii_uppercase:
        return ""
    local_index = ord(local_choice) - ord("A")
    if local_index < 0 or local_index >= len(option_indices):
        return ""
    return chr(ord("A") + option_indices[local_index])


def deterministic_letter_fallback(valid_letters, tie_break):
    if not valid_letters:
        return ""
    if tie_break == "reverse_alphabetical":
        return sorted(valid_letters, reverse=True)[0]
    return sorted(valid_letters)[0]


def parse_generation_index(path_value):
    if not path_value:
        return None
    match = re.search(r"gen_(\d+)", str(path_value))
    if match:
        return int(match.group(1))
    return None


def parse_generation_set(raw_value):
    if not raw_value:
        return set()
    generations = set()
    for chunk in raw_value.split(","):
        chunk = chunk.strip()
        if not chunk:
            continue
        if chunk.isdigit():
            generations.add(int(chunk))
    return generations


def make_invalid_metrics(protocol_mode, search_mini_size, error_type, error_message):
    metrics = {
        "search_score": 0.0,
        "search_correct": 0,
        "search_size": 0,
        "holdout_score": 0.0,
        "holdout_correct": 0,
        "holdout_size": 0,
        "combined_score": 0.0,
        "invalid_generation": 1,
        "error_type": error_type,
        "error_message": error_message,
        "search_mini_size": search_mini_size,
        "vlm_call_count": 0,
        "protocol_mode": protocol_mode,
        "split_manifest_version": "",
    }
    active_prefix = protocol_metric_prefix(protocol_mode)
    metrics[f"{active_prefix}_score"] = 0.0
    metrics[f"{active_prefix}_correct"] = 0
    metrics[f"{active_prefix}_size"] = 0
    return metrics


def safe_run_experiment(
    experiment_fn,
    protocol_mode,
    search_mini_size,
    task_name="unknown_task",
    results_dir=None,
    lock_mode=None,
):
    start_time = time.time()
    lock_info = {}
    try:
        with acquire_vlm_lock(
            task_name=task_name,
            results_dir=results_dir,
            mode=lock_mode or protocol_mode,
        ) as owner_info:
            lock_info = owner_info
            metrics = experiment_fn()
        metrics.setdefault("invalid_generation", 0)
        metrics.setdefault("error_type", "")
        metrics.setdefault("error_message", "")
    except Exception as exc:
        traceback.print_exc(file=sys.stderr)
        metrics = make_invalid_metrics(
            protocol_mode=protocol_mode,
            search_mini_size=search_mini_size,
            error_type=type(exc).__name__,
            error_message=str(exc),
        )
    metrics["wall_time_sec"] = round(time.time() - start_time, 3)
    metrics["vlm_lock_wait_sec"] = round(lock_info.get("waited_for_lock_sec", 0.0), 3)
    return metrics


def collect_generation_records(run_dir):
    run_dir = Path(run_dir)
    records = []
    for generation_dir in sorted(run_dir.glob("gen_*"), key=lambda path: parse_generation_index(path) or -1):
        metrics_path = generation_dir / "results" / "metrics.json"
        if metrics_path.exists():
            try:
                metrics = json.loads(metrics_path.read_text())
            except Exception as exc:
                metrics = {
                    "combined_score": 0.0,
                    "invalid_generation": 1,
                    "error_type": "UnreadableMetrics",
                    "error_message": str(exc),
                    "metrics_missing": 1,
                    "incomplete_generation": 1,
                }
        else:
            metrics = {
                "combined_score": 0.0,
                "invalid_generation": 1,
                "error_type": "MissingMetrics",
                "error_message": "metrics.json missing",
                "metrics_missing": 1,
                "incomplete_generation": 1,
            }
        records.append(
            {
                "generation": parse_generation_index(generation_dir),
                "generation_dir": str(generation_dir),
                "program_path": str(generation_dir / "main.py"),
                "metrics_path": str(metrics_path),
                **metrics,
            }
        )
    return records


def select_top_k_records(records, top_k=3):
    valid_records = [record for record in records if not record.get("invalid_generation")]
    ordered = sorted(
        valid_records,
        key=lambda record: (
            -record.get("combined_score", 0.0),
            record.get("generation", 10**9),
        ),
    )
    return ordered[:top_k]


def select_best_so_far(records, milestone_generation):
    eligible = [
        record
        for record in records
        if record.get("generation") is not None
        and record["generation"] <= milestone_generation
        and not record.get("invalid_generation")
    ]
    if not eligible:
        return None
    return sorted(
        eligible,
        key=lambda record: (
            -record.get("combined_score", 0.0),
            record.get("generation", 10**9),
        ),
    )[0]


In [ ]:
%%writefile create_medframeqa_split_manifest.py
import hashlib
import json
import math
import os
import random
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

from datasets import load_dataset


PROJECT_ROOT = Path("/gluon4/xl693/evolve")
DATASET_ID = os.environ.get("MEDFRAMEQA_DATASET", "SuhaoYu1020/MedFrameQA")
SOURCE_SPLIT = "test"
GROUP_KEY = "video_id"
MANIFEST_VERSION = "medframeqa_split_manifest_v2"
OUTPUT_PATH = PROJECT_ROOT / "medframeqa_split_manifest_v1.json"
DATASET_CACHE_DIR = os.environ.get("MEDFRAMEQA_DATASET_CACHE", "/tmp/medframeqa_hf_cache")

TARGET_SPLITS = {
    "evolution_pool": 1331,
    "selection_holdout": 665,
    "independent_final_test": 855,
}
SPLIT_NAMES = list(TARGET_SPLITS.keys())

MANDATORY_MODALITIES = ("CT", "MRI", "ultrasound", "X-ray")
HOLDOUT_OTHER_REQUIRED = ("selection_holdout", "independent_final_test")

OBJECTIVE_WEIGHTS = {
    "modality": 4.0,
    "answer": 1.5,
    "organ": 0.25,
}
HARD_PENALTIES = {
    "missing_primary_modality": 100.0,
    "missing_other": 50.0,
}

RESTART_COUNT = int(os.environ.get("MEDFRAMEQA_SPLIT_RESTARTS", "128"))
LOCAL_SEARCH_MAX_STEPS = int(os.environ.get("MEDFRAMEQA_SPLIT_LOCAL_STEPS", "24"))
PAIR_SIZE_LIMIT = int(os.environ.get("MEDFRAMEQA_SPLIT_BUNDLE_CAP", "12"))
GROUP_SHORTLIST = int(os.environ.get("MEDFRAMEQA_SPLIT_GROUP_SHORTLIST", "24"))
BASE_SEED = int(os.environ.get("MEDFRAMEQA_SPLIT_BASE_SEED", "135"))
GENERATOR_STRATEGY = "video_group_multistart_bundle_exchange_v2"


def _is_frame_image_column(column_name: str) -> bool:
    if not column_name.startswith("image_"):
        return False
    suffix = column_name.split("_", 1)[1]
    return suffix.isdigit()


@dataclass(frozen=True)
class GroupRecord:
    group_id: str
    question_ids: tuple
    size: int
    modality: tuple
    answer: tuple
    organ: tuple


@dataclass(frozen=True)
class BundleRecord:
    members: tuple
    size: int
    modality: tuple
    answer: tuple
    organ: tuple


def _stable_hash(text):
    return int(hashlib.sha256(text.encode("utf-8")).hexdigest()[:16], 16)


def _load_rows():
    dataset = load_dataset(
        DATASET_ID,
        split=SOURCE_SPLIT,
        cache_dir=DATASET_CACHE_DIR,
    )
    image_columns = [
        column
        for column in dataset.column_names
        if column == "image_url" or _is_frame_image_column(column)
    ]
    if image_columns:
        dataset = dataset.remove_columns(image_columns)
    return [dict(row) for row in dataset]


def _build_key_spaces(rows):
    modality_keys = sorted(Counter(row["modality"] for row in rows))
    answer_keys = sorted(Counter(row["correct_answer"] for row in rows))
    organ_counts = Counter(row["organ"] for row in rows)
    organ_keys = [organ for organ, _ in organ_counts.most_common(10)]
    return modality_keys, answer_keys, organ_keys


def _group_rows(rows, modality_keys, answer_keys, organ_keys):
    grouped = defaultdict(list)
    for row in rows:
        grouped[row[GROUP_KEY]].append(row)

    groups = []
    for group_id, members in grouped.items():
        question_ids = tuple(row["question_id"] for row in members)
        modality_counter = Counter(row["modality"] for row in members)
        answer_counter = Counter(row["correct_answer"] for row in members)
        organ_counter = Counter(row["organ"] for row in members)
        groups.append(
            GroupRecord(
                group_id=str(group_id),
                question_ids=question_ids,
                size=len(members),
                modality=tuple(modality_counter.get(key, 0) for key in modality_keys),
                answer=tuple(answer_counter.get(key, 0) for key in answer_keys),
                organ=tuple(organ_counter.get(key, 0) for key in organ_keys),
            )
        )
    groups.sort(key=lambda group: (_stable_hash(group.group_id), group.group_id))
    return groups


def _sum_vectors(groups, attribute, count):
    total = [0] * count
    for group in groups:
        vector = getattr(group, attribute)
        for index, value in enumerate(vector):
            total[index] += value
    return total


def _vector_from_counter(counter, keys):
    return [counter.get(key, 0) for key in keys]


def _expected_counts(total_vector, target_size, total_size):
    return [target_size * value / total_size for value in total_vector]


def _relative_deviation(actual, expected):
    deviations = []
    for actual_value, expected_value in zip(actual, expected):
        baseline = max(expected_value, 1e-9)
        deviations.append(abs(actual_value - expected_value) / baseline)
    return deviations


def _objective_from_vectors(split_name, modality, answer, organ, expected):
    modality_dev = _relative_deviation(modality, expected[split_name]["modality"])
    answer_dev = _relative_deviation(answer, expected[split_name]["answer"])
    organ_dev = _relative_deviation(organ, expected[split_name]["organ"])

    objective = 0.0
    objective += OBJECTIVE_WEIGHTS["modality"] * sum(modality_dev)
    objective += OBJECTIVE_WEIGHTS["answer"] * sum(answer_dev)
    objective += OBJECTIVE_WEIGHTS["organ"] * sum(organ_dev)

    modality_keys = expected["_modality_keys"]
    modality_index = {key: index for index, key in enumerate(modality_keys)}
    for key in MANDATORY_MODALITIES:
        index = modality_index.get(key)
        if index is not None and modality[index] <= 0:
            objective += HARD_PENALTIES["missing_primary_modality"]

    other_index = modality_index.get("other")
    if split_name in HOLDOUT_OTHER_REQUIRED and other_index is not None and modality[other_index] <= 0:
        objective += HARD_PENALTIES["missing_other"]
    return objective


def _assignment_objective(assignment, groups_by_id, expected):
    score = 0.0
    for split_name, group_ids in assignment.items():
        groups = [groups_by_id[group_id] for group_id in group_ids]
        modality = _sum_vectors(groups, "modality", len(expected["_modality_keys"]))
        answer = _sum_vectors(groups, "answer", len(expected["_answer_keys"]))
        organ = _sum_vectors(groups, "organ", len(expected["_organ_keys"]))
        score += _objective_from_vectors(split_name, modality, answer, organ, expected)
    return score


def _split_distribution_score(group, modality_target, answer_target, organ_target, seed_value):
    size = max(group.size, 1)
    modality_prop = [value / size for value in group.modality]
    answer_prop = [value / size for value in group.answer]
    organ_prop = [value / size for value in group.organ]

    modality_delta = sum(
        abs(modality_prop[index] - modality_target[index]) for index in range(len(modality_prop))
    )
    answer_delta = sum(abs(answer_prop[index] - answer_target[index]) for index in range(len(answer_prop)))
    organ_delta = sum(abs(organ_prop[index] - organ_target[index]) for index in range(len(organ_prop)))

    score = 0.0
    score += OBJECTIVE_WEIGHTS["modality"] * (2.0 - modality_delta)
    score += OBJECTIVE_WEIGHTS["answer"] * (2.0 - answer_delta)
    score += OBJECTIVE_WEIGHTS["organ"] * (2.0 - organ_delta)
    score *= size
    score += ((_stable_hash(f"{seed_value}:{group.group_id}") % 1_000_003) / 1_000_003.0) * 0.01
    return score


def _exact_knapsack_select(groups, target_size, values):
    neg_inf = -10**30
    dp = [(neg_inf, None)] * (target_size + 1)
    dp[0] = (0.0, tuple())

    for index, group in enumerate(groups):
        size = group.size
        value = values[index]
        next_dp = list(dp)
        for total in range(target_size, size - 1, -1):
            previous_score, previous_choice = dp[total - size]
            if previous_score <= neg_inf / 2:
                continue
            candidate = previous_score + value
            if candidate > next_dp[total][0]:
                next_dp[total] = (candidate, previous_choice + (index,))
        dp = next_dp

    score, chosen = dp[target_size]
    if chosen is None:
        raise ValueError(f"Could not solve exact knapsack for target size {target_size}")
    return set(chosen)


def _build_initial_assignment(groups, expected, seed_value):
    modality_target = {
        split_name: [
            value / TARGET_SPLITS[split_name] if TARGET_SPLITS[split_name] else 0.0
            for value in expected[split_name]["modality"]
        ]
        for split_name in SPLIT_NAMES
    }
    answer_target = {
        split_name: [
            value / TARGET_SPLITS[split_name] if TARGET_SPLITS[split_name] else 0.0
            for value in expected[split_name]["answer"]
        ]
        for split_name in SPLIT_NAMES
    }
    organ_target = {
        split_name: [
            value / TARGET_SPLITS[split_name] if TARGET_SPLITS[split_name] else 0.0
            for value in expected[split_name]["organ"]
        ]
        for split_name in SPLIT_NAMES
    }

    split_order = ["selection_holdout", "independent_final_test"]
    if seed_value % 2:
        split_order = ["independent_final_test", "selection_holdout"]

    for attempt in range(64):
        rng = random.Random(seed_value * 1009 + attempt)
        remaining = list(groups)
        rng.shuffle(remaining)
        assignment = {}

        try:
            for split_name in split_order:
                values = [
                    _split_distribution_score(
                        group,
                        modality_target[split_name],
                        answer_target[split_name],
                        organ_target[split_name],
                        f"{seed_value}:{attempt}:{split_name}",
                    )
                    for group in remaining
                ]
                chosen_indices = _exact_knapsack_select(remaining, TARGET_SPLITS[split_name], values)
                chosen_groups = [group for index, group in enumerate(remaining) if index in chosen_indices]
                assignment[split_name] = tuple(group.group_id for group in chosen_groups)
                remaining = [group for index, group in enumerate(remaining) if index not in chosen_indices]

            assignment["evolution_pool"] = tuple(group.group_id for group in remaining)
            return {split_name: tuple(sorted(group_ids)) for split_name, group_ids in assignment.items()}
        except ValueError:
            continue

    raise ValueError(f"Could not build an exact initial assignment for seed {seed_value}")


def _vector_add(left, right):
    return tuple(a + b for a, b in zip(left, right))


def _vector_sub(left, right):
    return tuple(a - b for a, b in zip(left, right))


def _build_split_state(assignment, groups_by_id, expected):
    states = {}
    for split_name, group_ids in assignment.items():
        groups = [groups_by_id[group_id] for group_id in group_ids]
        modality = tuple(_sum_vectors(groups, "modality", len(expected["_modality_keys"])))
        answer = tuple(_sum_vectors(groups, "answer", len(expected["_answer_keys"])))
        organ = tuple(_sum_vectors(groups, "organ", len(expected["_organ_keys"])))
        states[split_name] = {
            "group_ids": list(group_ids),
            "group_set": set(group_ids),
            "size": sum(group.size for group in groups),
            "modality": modality,
            "answer": answer,
            "organ": organ,
            "objective": _objective_from_vectors(split_name, modality, answer, organ, expected),
        }
    return states


def _make_bundle(group_ids, groups_by_id):
    groups = [groups_by_id[group_id] for group_id in group_ids]
    return BundleRecord(
        members=tuple(sorted(group_ids)),
        size=sum(group.size for group in groups),
        modality=tuple(_sum_vectors(groups, "modality", len(groups[0].modality))),
        answer=tuple(_sum_vectors(groups, "answer", len(groups[0].answer))),
        organ=tuple(_sum_vectors(groups, "organ", len(groups[0].organ))),
    )


def _bundle_direction_score(bundle, source_state, target_state, source_name, target_name, expected):
    score = 0.0
    for weight_name, attribute in (("modality", "modality"), ("answer", "answer"), ("organ", "organ")):
        weight = OBJECTIVE_WEIGHTS[weight_name]
        bundle_vector = getattr(bundle, attribute)
        source_actual = source_state[attribute]
        target_actual = target_state[attribute]
        source_expected = expected[source_name][weight_name]
        target_expected = expected[target_name][weight_name]
        for index, value in enumerate(bundle_vector):
            if value <= 0:
                continue
            source_baseline = max(source_expected[index], 1e-9)
            target_baseline = max(target_expected[index], 1e-9)
            source_surplus = max(0.0, source_actual[index] - source_expected[index]) / source_baseline
            target_deficit = max(0.0, target_expected[index] - target_actual[index]) / target_baseline
            score += weight * value * (source_surplus + target_deficit)
    return score


def _candidate_bundles(group_ids, groups_by_id):
    indexed = defaultdict(list)
    sorted_ids = sorted(group_ids)
    for group_id in sorted_ids:
        bundle = _make_bundle((group_id,), groups_by_id)
        indexed[bundle.size].append(bundle)
    for left_index, left_group in enumerate(sorted_ids):
        left_size = groups_by_id[left_group].size
        for right_group in sorted_ids[left_index + 1 :]:
            bundle = _make_bundle((left_group, right_group), groups_by_id)
            indexed[bundle.size].append(bundle)
    return indexed


def _shortlist_group_ids(group_ids, groups_by_id, source_state, target_state, source_name, target_name, expected):
    scored = []
    for group_id in group_ids:
        bundle = _make_bundle((group_id,), groups_by_id)
        score = _bundle_direction_score(bundle, source_state, target_state, source_name, target_name, expected)
        scored.append((score, groups_by_id[group_id].size, group_id))
    scored.sort(key=lambda item: (-item[0], -item[1], item[2]))
    return [group_id for _, _, group_id in scored[:GROUP_SHORTLIST]]


def _rank_bundles(bundle_index, source_state, target_state, source_name, target_name, expected, bundle_cap):
    ranked = {}
    for size, bundles in bundle_index.items():
        scored = [
            (
                _bundle_direction_score(bundle, source_state, target_state, source_name, target_name, expected),
                bundle,
            )
            for bundle in bundles
        ]
        scored.sort(key=lambda item: (-item[0], item[1].members))
        ranked[size] = [bundle for _, bundle in scored[:bundle_cap]]
    return ranked


def _apply_exchange(states, split_a, split_b, bundle_a, bundle_b, expected):
    state_a = states[split_a]
    state_b = states[split_b]

    new_modality_a = _vector_add(_vector_sub(state_a["modality"], bundle_a.modality), bundle_b.modality)
    new_answer_a = _vector_add(_vector_sub(state_a["answer"], bundle_a.answer), bundle_b.answer)
    new_organ_a = _vector_add(_vector_sub(state_a["organ"], bundle_a.organ), bundle_b.organ)

    new_modality_b = _vector_add(_vector_sub(state_b["modality"], bundle_b.modality), bundle_a.modality)
    new_answer_b = _vector_add(_vector_sub(state_b["answer"], bundle_b.answer), bundle_a.answer)
    new_organ_b = _vector_add(_vector_sub(state_b["organ"], bundle_b.organ), bundle_a.organ)

    state_a["modality"] = new_modality_a
    state_a["answer"] = new_answer_a
    state_a["organ"] = new_organ_a
    state_b["modality"] = new_modality_b
    state_b["answer"] = new_answer_b
    state_b["organ"] = new_organ_b
    state_a["objective"] = _objective_from_vectors(split_a, new_modality_a, new_answer_a, new_organ_a, expected)
    state_b["objective"] = _objective_from_vectors(split_b, new_modality_b, new_answer_b, new_organ_b, expected)

    for member in bundle_a.members:
        state_a["group_set"].remove(member)
        state_b["group_set"].add(member)
    for member in bundle_b.members:
        state_b["group_set"].remove(member)
        state_a["group_set"].add(member)
    state_a["group_ids"] = sorted(state_a["group_set"])
    state_b["group_ids"] = sorted(state_b["group_set"])


def _local_search(assignment, groups_by_id, expected):
    states = _build_split_state(assignment, groups_by_id, expected)
    for _ in range(LOCAL_SEARCH_MAX_STEPS):
        split_order = sorted(SPLIT_NAMES, key=lambda name: states[name]["objective"], reverse=True)
        best_move = None
        best_delta = 0.0

        for left_index, split_a in enumerate(split_order):
            for split_b in split_order[left_index + 1 :]:
                state_a = states[split_a]
                state_b = states[split_b]
                shortlist_a = _shortlist_group_ids(
                    state_a["group_ids"], groups_by_id, state_a, state_b, split_a, split_b, expected
                )
                shortlist_b = _shortlist_group_ids(
                    state_b["group_ids"], groups_by_id, state_b, state_a, split_b, split_a, expected
                )
                bundle_index_a = _candidate_bundles(shortlist_a, groups_by_id)
                bundle_index_b = _candidate_bundles(shortlist_b, groups_by_id)
                ranked_a = _rank_bundles(
                    bundle_index_a, state_a, state_b, split_a, split_b, expected, PAIR_SIZE_LIMIT
                )
                ranked_b = _rank_bundles(
                    bundle_index_b, state_b, state_a, split_b, split_a, expected, PAIR_SIZE_LIMIT
                )

                common_sizes = set(ranked_a).intersection(ranked_b)
                pair_before = state_a["objective"] + state_b["objective"]
                for size in sorted(common_sizes):
                    for bundle_a in ranked_a[size]:
                        for bundle_b in ranked_b[size]:
                            if set(bundle_a.members) & set(bundle_b.members):
                                continue
                            modality_a = _vector_add(_vector_sub(state_a["modality"], bundle_a.modality), bundle_b.modality)
                            answer_a = _vector_add(_vector_sub(state_a["answer"], bundle_a.answer), bundle_b.answer)
                            organ_a = _vector_add(_vector_sub(state_a["organ"], bundle_a.organ), bundle_b.organ)
                            modality_b = _vector_add(_vector_sub(state_b["modality"], bundle_b.modality), bundle_a.modality)
                            answer_b = _vector_add(_vector_sub(state_b["answer"], bundle_b.answer), bundle_a.answer)
                            organ_b = _vector_add(_vector_sub(state_b["organ"], bundle_b.organ), bundle_a.organ)
                            pair_after = _objective_from_vectors(
                                split_a, modality_a, answer_a, organ_a, expected
                            ) + _objective_from_vectors(split_b, modality_b, answer_b, organ_b, expected)
                            delta = pair_before - pair_after
                            if delta > best_delta + 1e-9:
                                best_delta = delta
                                best_move = (split_a, split_b, bundle_a, bundle_b)

        if not best_move:
            break
        _apply_exchange(states, *best_move, expected)

    return {split_name: tuple(states[split_name]["group_ids"]) for split_name in SPLIT_NAMES}


def _compute_expected(rows, modality_keys, answer_keys, organ_keys):
    total_rows = len(rows)
    modality_total = _vector_from_counter(Counter(row["modality"] for row in rows), modality_keys)
    answer_total = _vector_from_counter(Counter(row["correct_answer"] for row in rows), answer_keys)
    organ_total = _vector_from_counter(Counter(row["organ"] for row in rows), organ_keys)

    expected = {
        "_modality_keys": list(modality_keys),
        "_answer_keys": list(answer_keys),
        "_organ_keys": list(organ_keys),
    }
    for split_name, target_size in TARGET_SPLITS.items():
        expected[split_name] = {
            "modality": _expected_counts(modality_total, target_size, total_rows),
            "answer": _expected_counts(answer_total, target_size, total_rows),
            "organ": _expected_counts(organ_total, target_size, total_rows),
        }
    return expected


def _split_stats(split_name, group_ids, groups_by_id, expected):
    groups = [groups_by_id[group_id] for group_id in group_ids]
    size = sum(group.size for group in groups)
    modality_actual = _sum_vectors(groups, "modality", len(expected["_modality_keys"]))
    answer_actual = _sum_vectors(groups, "answer", len(expected["_answer_keys"]))
    organ_actual = _sum_vectors(groups, "organ", len(expected["_organ_keys"]))
    modality_expected = expected[split_name]["modality"]
    answer_expected = expected[split_name]["answer"]
    organ_expected = expected[split_name]["organ"]
    modality_deviation = _relative_deviation(modality_actual, modality_expected)
    answer_deviation = _relative_deviation(answer_actual, answer_expected)
    organ_deviation = _relative_deviation(organ_actual, organ_expected)

    modality_map = {key: modality_actual[index] for index, key in enumerate(expected["_modality_keys"])}
    answer_map = {key: answer_actual[index] for index, key in enumerate(expected["_answer_keys"])}
    organ_map = {key: organ_actual[index] for index, key in enumerate(expected["_organ_keys"])}
    expected_modality_map = {
        key: round(modality_expected[index], 3) for index, key in enumerate(expected["_modality_keys"])
    }
    expected_answer_map = {
        key: round(answer_expected[index], 3) for index, key in enumerate(expected["_answer_keys"])
    }
    expected_organ_map = {
        key: round(organ_expected[index], 3) for index, key in enumerate(expected["_organ_keys"])
    }
    modality_dev_map = {
        key: round(modality_deviation[index], 4) for index, key in enumerate(expected["_modality_keys"])
    }
    answer_dev_map = {
        key: round(answer_deviation[index], 4) for index, key in enumerate(expected["_answer_keys"])
    }
    organ_dev_map = {key: round(organ_deviation[index], 4) for index, key in enumerate(expected["_organ_keys"])}

    def _summary(values):
        return {
            "mean": round(sum(values) / len(values), 4) if values else 0.0,
            "max": round(max(values), 4) if values else 0.0,
        }

    return {
        "size": size,
        "group_count": len(group_ids),
        "answer_distribution": answer_map,
        "modality_distribution": modality_map,
        "top_organs": organ_map,
        "expected_answer_distribution": expected_answer_map,
        "expected_modality_distribution": expected_modality_map,
        "expected_top_organs": expected_organ_map,
        "answer_relative_deviation": answer_dev_map,
        "modality_relative_deviation": modality_dev_map,
        "top_organ_relative_deviation": organ_dev_map,
        "relative_deviation_summary": {
            "answer": _summary(answer_deviation),
            "modality": _summary(modality_deviation),
            "organ": _summary(organ_deviation),
        },
        "objective_score": round(
            _objective_from_vectors(split_name, tuple(modality_actual), tuple(answer_actual), tuple(organ_actual), expected),
            6,
        ),
    }


def _validate_assignment(assignment, groups_by_id):
    all_question_ids = []
    seen_groups = set()
    for split_name, target_size in TARGET_SPLITS.items():
        group_ids = assignment[split_name]
        groups = [groups_by_id[group_id] for group_id in group_ids]
        split_size = sum(group.size for group in groups)
        if split_size != target_size:
            raise ValueError(f"{split_name} size mismatch: {split_size} != {target_size}")
        for group_id in group_ids:
            if group_id in seen_groups:
                raise ValueError(f"group leakage detected for {group_id}")
            seen_groups.add(group_id)
        for group in groups:
            all_question_ids.extend(group.question_ids)
    if len(all_question_ids) != len(set(all_question_ids)):
        raise ValueError("question_id overlap detected across splits")


def _manifest_payload(rows, groups, assignment, expected, seed_value, objective_score):
    groups_by_id = {group.group_id: group for group in groups}
    stats = {
        split_name: _split_stats(split_name, assignment[split_name], groups_by_id, expected)
        for split_name in SPLIT_NAMES
    }

    payload = {
        "version": MANIFEST_VERSION,
        "dataset_id": DATASET_ID,
        "source_split": SOURCE_SPLIT,
        "group_key": GROUP_KEY,
        "targets": TARGET_SPLITS,
        "generator": {
            "seed": seed_value,
            "base_seed": BASE_SEED,
            "strategy": GENERATOR_STRATEGY,
            "restart_count": RESTART_COUNT,
            "local_search_max_steps": LOCAL_SEARCH_MAX_STEPS,
            "group_shortlist": GROUP_SHORTLIST,
            "bundle_cap": PAIR_SIZE_LIMIT,
            "objective_weights": OBJECTIVE_WEIGHTS,
            "hard_penalties": HARD_PENALTIES,
            "bundle_exchange": ["1<->1", "1<->2", "2<->1", "2<->2"],
        },
        "total_examples": len(rows),
        "splits": {
            split_name: [
                question_id
                for group_id in assignment[split_name]
                for question_id in groups_by_id[group_id].question_ids
            ]
            for split_name in SPLIT_NAMES
        },
        "stats": stats,
        "objective_score": round(objective_score, 6),
    }
    manifest_text = json.dumps(payload, indent=2, ensure_ascii=False, sort_keys=True)
    payload["manifest_sha256"] = hashlib.sha256(manifest_text.encode("utf-8")).hexdigest()
    return payload


def _run_search(rows):
    modality_keys, answer_keys, organ_keys = _build_key_spaces(rows)
    groups = _group_rows(rows, modality_keys, answer_keys, organ_keys)
    groups_by_id = {group.group_id: group for group in groups}
    expected = _compute_expected(rows, modality_keys, answer_keys, organ_keys)

    best_assignment = None
    best_score = math.inf
    best_seed = None

    for restart_index in range(RESTART_COUNT):
        seed_value = BASE_SEED + restart_index
        assignment = _build_initial_assignment(groups, expected, seed_value)
        assignment = _local_search(assignment, groups_by_id, expected)
        _validate_assignment(assignment, groups_by_id)
        score = _assignment_objective(assignment, groups_by_id, expected)
        if score < best_score:
            best_score = score
            best_assignment = assignment
            best_seed = seed_value

    return groups, best_assignment, expected, best_seed, best_score


def _needs_regeneration(path):
    if not path.exists():
        return True
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return True
    if payload.get("version") != MANIFEST_VERSION:
        return True
    generator = payload.get("generator", {})
    if generator.get("strategy") != GENERATOR_STRATEGY:
        return True
    return False


def ensure_split_manifest(output_path=None, force=False):
    path = Path(output_path) if output_path else OUTPUT_PATH
    if not force and not _needs_regeneration(path):
        return path

    rows = _load_rows()
    groups, assignment, expected, best_seed, best_score = _run_search(rows)
    payload = _manifest_payload(rows, groups, assignment, expected, best_seed, best_score)

    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n")
    return path


if __name__ == "__main__":
    output_path = ensure_split_manifest(force=True)
    print(f"Wrote manifest to {output_path}")


In [ ]:
import os
os.makedirs("advanced_vqa_task_order_vote_plus", exist_ok=True)


In [ ]:
%%writefile advanced_vqa_task_order_vote_plus/initial.py
"""
order_vote_plus：顺序投票 + 条件 top-2 纠错。

整体思路：
- 第一阶段沿用多个 option order 的直接投票；
- 只有在投票不稳定时，才触发一次 top-2 定向纠错；
- 可进化空间仍然限制在少量文本槽位和离散开关上。
"""

from pathlib import Path
import sys


def _ensure_project_root():
    path = Path(__file__).resolve()
    candidates = [path.parent, *path.parents]
    for candidate in candidates:
        if (candidate / "medframeqa_runtime.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    return None


_ensure_project_root()

from medframeqa_runtime import (
    build_content_list,
    coerce_choice,
    coerce_int_choice,
    format_options,
    infer_case_metadata,
    merge_text_config,
    parse_json_config_block,
    render_metadata_block,
)


DEFAULT_PROMPT_CONFIG = {
"system_prompts": {
    "direct": "You are a radiologist choosing the best answer to a multi-image medical multiple-choice question. Return only the local choice requested.\n",
    "top2_rerank": "You are a radiologist resolving an uncertain vote between two shortlisted answers. Return only the better local choice requested.\n"
},
"direct_instruction": "Review the full image sequence and choose the best option for this vote. The options may be re-ordered; judge option content rather than fixed letter position.\n",
"sequence_instruction": "Prefer the answer that best explains the full ordered sequence, not a single frame.\n",
"metadata_instruction": "Use the metadata anchors only to reject options that clearly conflict with modality, organ, or sequence.\n",
"top2_rerank_prompt": "The first-stage vote is uncertain. Compare only local option A and local option B, then return the better local choice.\n",
"decision_prefixes": {
    "direct": "Best local option: ",
    "top2_rerank": "Better local option: "
},
"image_prompt_template": "[IMAGE {index}/{total}] Checklist: anatomy | key finding | temporal role | option impact.",
"order_views": 2,
"vote_tie_break": "alphabetical",
"uncertainty_trigger_rule": "margin_or_tie_or_all_disagree",
"fallback_rule": "vote_winner"
}


# EVOLVE-BLOCK-START
# 说明：只编辑下面 JSON 文本中的值。
PROMPT_CONFIG_JSON = r'''
{
  "system_prompts": {
"direct": "You are a radiologist choosing the best answer to a multi-image medical multiple-choice question. Return only the local choice requested.\n",
"top2_rerank": "You are a radiologist resolving an uncertain vote between two shortlisted answers. Return only the better local choice requested.\n"
  },
  "direct_instruction": "Review the full image sequence and choose the best option for this vote. The options may be re-ordered; judge option content rather than fixed letter position.\n",
  "sequence_instruction": "Prefer the answer that best explains the full ordered sequence, not a single frame.\n",
  "metadata_instruction": "Use the metadata anchors only to reject options that clearly conflict with modality, organ, or sequence.\n",
  "top2_rerank_prompt": "The first-stage vote is uncertain. Compare only local option A and local option B, then return the better local choice.\n",
  "decision_prefixes": {
"direct": "Best local option: ",
"top2_rerank": "Better local option: "
  },
  "image_prompt_template": "[IMAGE {index}/{total}] Checklist: anatomy | key finding | temporal role | option impact.",
  "order_views": 2,
  "vote_tie_break": "alphabetical",
  "uncertainty_trigger_rule": "margin_or_tie_or_all_disagree",
  "fallback_rule": "vote_winner"
}
'''
# EVOLVE-BLOCK-END


def generate_prompt_config():
    return parse_json_config_block(PROMPT_CONFIG_JSON)


def get_prompt_config():
    candidate = generate_prompt_config()
    merged = merge_text_config(DEFAULT_PROMPT_CONFIG, candidate)
    merged["order_views"] = coerce_int_choice(
        candidate.get("order_views"),
        {1, 2},
        DEFAULT_PROMPT_CONFIG["order_views"],
    )
    merged["vote_tie_break"] = coerce_choice(
        candidate.get("vote_tie_break"),
        {"alphabetical", "reverse_alphabetical"},
        DEFAULT_PROMPT_CONFIG["vote_tie_break"],
    )
    merged["uncertainty_trigger_rule"] = coerce_choice(
        candidate.get("uncertainty_trigger_rule"),
        {"margin_or_tie_or_all_disagree", "margin_or_tie", "all_disagree_only"},
        DEFAULT_PROMPT_CONFIG["uncertainty_trigger_rule"],
    )
    merged["fallback_rule"] = coerce_choice(
        candidate.get("fallback_rule"),
        {"vote_winner", "top2_tie_break"},
        DEFAULT_PROMPT_CONFIG["fallback_rule"],
    )
    return merged


def get_runtime_config():
    config = get_prompt_config()
    return {
        "task_family": "order_vote_plus",
        "order_views": config["order_views"],
        "vote_tie_break": config["vote_tie_break"],
        "uncertainty_trigger_rule": config["uncertainty_trigger_rule"],
        "fallback_rule": config["fallback_rule"],
    }


def format_vqa_payload(sample, mode="direct", option_indices=None, order_name="identity"):
    config = get_prompt_config()
    metadata_block = render_metadata_block(infer_case_metadata(sample))

    if option_indices is None:
        option_indices = list(range(len(sample["options"])))

    if mode == "direct":
        prompt_text = (
            f"{config['direct_instruction']}"
            f"{config['sequence_instruction']}"
            f"{config['metadata_instruction']}\n"
            f"Current option ordering for this vote: {order_name}. "
            "Treat the displayed letters as local labels only.\n\n"
            f"{metadata_block}\n"
            f"Question:\n{sample['question']}\n\n"
            f"Options for this vote:\n"
            f"{format_options(sample['options'], option_indices=option_indices)}\n"
        )
        system_prompt = config["system_prompts"]["direct"]
        answer_prefix = config["decision_prefixes"]["direct"]
    elif mode == "top2_rerank":
        if len(option_indices) != 2:
            raise ValueError("top2_rerank 模式只允许 2 个候选")
        prompt_text = (
            f"{config['top2_rerank_prompt']}"
            f"{config['sequence_instruction']}"
            f"{config['metadata_instruction']}\n"
            f"Current shortlist order: {order_name}. "
            "Judge only local option A and local option B.\n\n"
            f"{metadata_block}\n"
            f"Question:\n{sample['question']}\n\n"
            f"Shortlisted options:\n"
            f"{format_options(sample['options'], option_indices=option_indices)}\n"
        )
        system_prompt = config["system_prompts"]["top2_rerank"]
        answer_prefix = config["decision_prefixes"]["top2_rerank"]
    else:
        raise ValueError(f"Unsupported mode for order_vote_plus: {mode}")

    content_list, _ = build_content_list(
        sample,
        prompt_text,
        config["image_prompt_template"],
    )
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": content_list},
        {"role": "assistant", "content": answer_prefix},
    ]


In [ ]:
%%writefile advanced_vqa_task_order_vote_plus/evaluate.py
"""
order_vote_plus 评测器。

整体思路：
- 第一阶段沿用 order_vote 的多顺序直接投票；
- 只有在投票不稳定时，才触发一次 top-2 定向纠错；
- 保持统一 metrics schema，只额外记录 uncertainty_trigger_count。
"""

from pathlib import Path
import sys


def _ensure_project_root():
    path = Path(__file__).resolve()
    candidates = [path.parent, *path.parents]
    for candidate in candidates:
        if (candidate / "medframeqa_runtime.py").exists():
            candidate_str = str(candidate)
            if candidate_str not in sys.path:
                sys.path.insert(0, candidate_str)
            return candidate
    return None


_ensure_project_root()

import argparse
import json
import os
import string
import sys
from collections import Counter
from pathlib import Path

from shinka.core import run_shinka_eval

from medframeqa_runtime import (
    deterministic_letter_fallback,
    extract_option_letter,
    generate_guided_choice_with_retries,
    get_option_orders,
    get_protocol_meta,
    get_protocol_score_key,
    get_protocol_subset,
    load_medframeqa_dataset,
    load_mutated_module,
    load_split_manifest,
    local_to_global,
    make_openai_client,
    make_protocol_metrics,
    parse_generation_index,
    parse_generation_set,
    safe_run_experiment,
    write_json,
)


TASK_NAME = "advanced_vqa_task_order_vote_plus"
VLM_URL = os.environ.get("MEDFRAMEQA_VLM_URL", "http://localhost:8001/v1")
VLM_MODEL = os.environ.get("MEDFRAMEQA_VLM_MODEL", "cyankiwi/Qwen3.5-35B-A3B-AWQ-4bit")
DEFAULT_PROTOCOL_MODE = os.environ.get("MEDFRAMEQA_PROTOCOL_MODE", "search_mini")
DEFAULT_SEARCH_MINI_SIZE = int(os.environ.get("MEDFRAMEQA_SEARCH_MINI_SIZE", "256"))
POOL_REEVAL_GENS = parse_generation_set(os.environ.get("MEDFRAMEQA_POOL_REEVAL_GENS", "10,20,30,40,49"))
API_TIMEOUT = float(os.environ.get("MEDFRAMEQA_API_TIMEOUT", "30"))


def choose_vote_winner(valid_letters, direct_votes, tie_break):
    if tie_break == "reverse_alphabetical":
        ranked = sorted(valid_letters, key=lambda letter: (-direct_votes[letter], -ord(letter)))
    else:
        ranked = sorted(valid_letters, key=lambda letter: (-direct_votes[letter], letter))
    return ranked[0] if ranked else ""


def rank_vote_letters(valid_letters, direct_votes, tie_break):
    if tie_break == "reverse_alphabetical":
        return sorted(valid_letters, key=lambda letter: (-direct_votes[letter], -ord(letter)))
    return sorted(valid_letters, key=lambda letter: (-direct_votes[letter], letter))


def should_trigger_uncertainty(direct_votes, ranked_letters, trigger_rule):
    voted_letters = [letter for letter in ranked_letters if direct_votes.get(letter, 0) > 0]
    total_votes = sum(direct_votes.values())
    if total_votes <= 1 or len(voted_letters) < 2:
        return False

    top1 = direct_votes.get(voted_letters[0], 0)
    top2 = direct_votes.get(voted_letters[1], 0)
    margin = top1 - top2
    highest_tie_count = sum(1 for letter in voted_letters if direct_votes.get(letter, 0) == top1)
    all_disagree = len(voted_letters) == total_votes and all(direct_votes.get(letter, 0) == 1 for letter in voted_letters)

    if trigger_rule == "all_disagree_only":
        return all_disagree
    if trigger_rule == "margin_or_tie":
        return margin <= 1 or highest_tie_count >= 2
    return margin <= 1 or highest_tie_count >= 2 or all_disagree


def apply_top2_fallback(top2_letters, direct_votes, tie_break, fallback_rule, vote_winner):
    if fallback_rule == "top2_tie_break" and top2_letters:
        top2_votes = Counter({letter: direct_votes.get(letter, 0) for letter in top2_letters})
        pred = choose_vote_winner(top2_letters, top2_votes, tie_break)
        if pred:
            return pred
    return vote_winner


def evaluate_sample(module, sample, client, call_stats):
    format_fn = module.format_vqa_payload
    runtime_config = module.get_runtime_config() if hasattr(module, "get_runtime_config") else {}
    valid_letters = list(string.ascii_uppercase[: len(sample["options"])])
    gt = sample["correct_answer"].strip().upper()
    order_views = runtime_config.get("order_views", 2)
    tie_break = runtime_config.get("vote_tie_break", "alphabetical")
    trigger_rule = runtime_config.get("uncertainty_trigger_rule", "margin_or_tie_or_all_disagree")
    fallback_rule = runtime_config.get("fallback_rule", "vote_winner")

    direct_votes = Counter()
    direct_debug = []
    missing_vote_count = 0

    for order_name, option_indices in get_option_orders(len(sample["options"]), order_views):
        local_letters = list(string.ascii_uppercase[: len(option_indices)])
        raw = generate_guided_choice_with_retries(
            client,
            lambda option_indices=option_indices, order_name=order_name: format_fn(
                sample,
                mode="direct",
                option_indices=option_indices,
                order_name=order_name,
            ),
            local_letters,
            VLM_MODEL,
            f"[direct] QID={sample.get('question_id', 'UNKNOWN')} order={order_name}",
            call_stats=call_stats,
        )
        local_pred = extract_option_letter(raw, local_letters)
        global_pred = local_to_global(local_pred, option_indices)
        if global_pred:
            direct_votes[global_pred] += 1
        else:
            missing_vote_count += 1
        direct_debug.append(
            {
                "order_name": order_name,
                "raw": raw,
                "local_pred": local_pred,
                "global_pred": global_pred,
            }
        )

    ranked_letters = rank_vote_letters(valid_letters, direct_votes, tie_break)
    vote_winner = choose_vote_winner(valid_letters, direct_votes, tie_break)
    if not vote_winner:
        vote_winner = deterministic_letter_fallback(valid_letters, tie_break)

    uncertainty_triggered = 0
    top2_rerank_debug = {}
    pred = vote_winner
    rerank_fallback_count = 0

    if should_trigger_uncertainty(direct_votes, ranked_letters, trigger_rule):
        top2_letters = [letter for letter in ranked_letters if direct_votes.get(letter, 0) > 0][:2]
        if len(top2_letters) == 2:
            uncertainty_triggered = 1
            option_indices = [ord(letter) - ord("A") for letter in top2_letters]
            local_letters = list(string.ascii_uppercase[: len(option_indices)])
            raw = generate_guided_choice_with_retries(
                client,
                lambda option_indices=option_indices: format_fn(
                    sample,
                    mode="top2_rerank",
                    option_indices=option_indices,
                    order_name="uncertain_top2",
                ),
                local_letters,
                VLM_MODEL,
                f"[top2_rerank] QID={sample.get('question_id', 'UNKNOWN')}",
                call_stats=call_stats,
            )
            local_pred = extract_option_letter(raw, local_letters)
            rerank_pred = local_to_global(local_pred, option_indices)
            if rerank_pred:
                pred = rerank_pred
            else:
                rerank_fallback_count = 1
                pred = apply_top2_fallback(
                    top2_letters,
                    direct_votes,
                    tie_break,
                    fallback_rule,
                    vote_winner,
                )
            top2_rerank_debug = {
                "top2_letters": top2_letters,
                "raw": raw,
                "local_pred": local_pred,
                "rerank_pred": rerank_pred,
            }

    if not pred:
        pred = deterministic_letter_fallback(valid_letters, tie_break)

    return {
        "gt": gt,
        "pred": pred,
        "direct_votes": dict(direct_votes),
        "direct_debug": direct_debug,
        "missing_vote_count": missing_vote_count,
        "all_votes_missing": int(sum(direct_votes.values()) == 0),
        "uncertainty_triggered": uncertainty_triggered,
        "rerank_fallback_count": rerank_fallback_count,
        "top2_rerank_debug": top2_rerank_debug,
    }


def evaluate_subset(module, subset, split_name, client, call_stats):
    correct = 0
    total = len(subset)
    missing_vote_count = 0
    all_votes_missing_count = 0
    uncertainty_trigger_count = 0
    rerank_fallback_count = 0

    for sample in subset:
        result = evaluate_sample(module, sample, client, call_stats)
        missing_vote_count += result["missing_vote_count"]
        all_votes_missing_count += result["all_votes_missing"]
        uncertainty_trigger_count += result["uncertainty_triggered"]
        rerank_fallback_count += result["rerank_fallback_count"]
        print(
            f"[{split_name}] QID={sample.get('question_id', 'UNKNOWN')} "
            f"pred={result['pred']!r} gt={result['gt']!r} direct={result['direct_votes']!r} "
            f"uncertain={result['uncertainty_triggered']}",
            file=sys.stderr,
        )
        if result["pred"] == result["gt"]:
            correct += 1

    metrics = make_protocol_metrics(split_name, correct, total)
    metrics["missing_vote_count"] = missing_vote_count
    metrics["all_votes_missing_count"] = all_votes_missing_count
    metrics["uncertainty_trigger_count"] = uncertainty_trigger_count
    metrics["rerank_fallback_count"] = rerank_fallback_count
    return metrics


def format_feedback_score(value):
    if value is None:
        return "N/A"
    return f"{float(value):.4f}"


def build_text_feedback(metrics):
    lines = [
        f"search_score={format_feedback_score(metrics.get('search_score'))}",
        f"pool_score={format_feedback_score(metrics.get('evolution_pool_score'))}",
        f"invalid_generation={int(metrics.get('invalid_generation', 0))}",
        f"error_type={metrics.get('error_type') or 'none'}",
        f"missing_vote_count={int(metrics.get('missing_vote_count', 0) or 0)}",
        f"uncertainty_trigger_count={int(metrics.get('uncertainty_trigger_count', 0) or 0)}",
        f"rerank_fallback_count={int(metrics.get('rerank_fallback_count', 0) or 0)}",
    ]
    if metrics.get("invalid_generation"):
        lines.append(
            "The candidate failed during evaluation. Preserve the config schema and keep the top-2 correction path minimal and valid."
        )
        if metrics.get("error_message"):
            lines.append(f"error_message={metrics['error_message']}")
        return "\n".join(lines)

    lines.append(
        "Use top-2 correction only for genuinely uncertain votes, and keep the first-stage order-vote stable across option permutations."
    )
    if metrics.get("missing_vote_count", 0) > 0:
        lines.append(
            "Some local votes were empty or malformed. Tighten the direct vote prefix and keep output to one local letter."
        )
    if metrics.get("rerank_fallback_count", 0) > 0:
        lines.append(
            "Some triggered top-2 corrections fell back instead of resolving uncertainty. Make the top-2 rerank prompt more decisive."
        )
    pool_score = metrics.get("evolution_pool_score")
    search_score = metrics.get("search_score")
    if pool_score is not None and search_score is not None and pool_score + 1e-9 < search_score:
        lines.append(
            "The search-mini score is higher than the pool score. Reduce overfitting to a narrow subset and simplify the correction trigger."
        )
    return "\n".join(lines)


def _run(program_path, results_dir=None, protocol_mode=None, search_mini_size=None):
    protocol_mode = protocol_mode or DEFAULT_PROTOCOL_MODE
    search_mini_size = int(search_mini_size or DEFAULT_SEARCH_MINI_SIZE)
    module = load_mutated_module(program_path)
    dataset = load_medframeqa_dataset(include_images=True)
    manifest = load_split_manifest()
    subset, selected_ids = get_protocol_subset(dataset, manifest, protocol_mode, search_mini_size)
    client = make_openai_client(VLM_URL, timeout=API_TIMEOUT)
    call_stats = {"vlm_call_count": 0}

    metrics = evaluate_subset(module, subset, protocol_mode, client, call_stats)
    metrics.update(get_protocol_meta(manifest, protocol_mode, selected_ids, search_mini_size))

    generation_index = parse_generation_index(results_dir or program_path)
    if protocol_mode == "search_mini" and generation_index in POOL_REEVAL_GENS:
        pool_subset, _ = get_protocol_subset(dataset, manifest, "evolution_pool", search_mini_size)
        metrics.update(evaluate_subset(module, pool_subset, "evolution_pool", client, call_stats))

    metrics["vlm_call_count"] = call_stats["vlm_call_count"]
    metrics["combined_score"] = metrics[get_protocol_score_key(protocol_mode)]
    return metrics


def run_experiment(program_path, results_dir=None, protocol_mode=None, search_mini_size=None, **kwargs):
    chosen_mode = protocol_mode or DEFAULT_PROTOCOL_MODE
    chosen_search_mini_size = int(search_mini_size or DEFAULT_SEARCH_MINI_SIZE)
    metrics = safe_run_experiment(
        lambda: _run(
            program_path=program_path,
            results_dir=results_dir,
            protocol_mode=chosen_mode,
            search_mini_size=chosen_search_mini_size,
        ),
        chosen_mode,
        chosen_search_mini_size,
        task_name=TASK_NAME,
        results_dir=results_dir,
    )
    metrics["text_feedback"] = build_text_feedback(metrics)
    return metrics


def aggregate_fn(results):
    metrics = dict(results[0])
    metrics["text_feedback"] = build_text_feedback(metrics)
    return metrics


def main(program_path, results_dir):
    run_shinka_eval(
        program_path=os.path.abspath(__file__),
        results_dir=results_dir,
        experiment_fn_name="run_experiment",
        num_runs=1,
        get_experiment_kwargs=lambda idx: {
            "program_path": program_path,
            "results_dir": results_dir,
            "protocol_mode": DEFAULT_PROTOCOL_MODE,
            "search_mini_size": DEFAULT_SEARCH_MINI_SIZE,
        },
        aggregate_metrics_fn=aggregate_fn,
    )


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--program_path", required=True)
    parser.add_argument("--results_dir", required=True)
    parser.add_argument("--single_run", action="store_true")
    parser.add_argument("--protocol_mode", default=None)
    parser.add_argument("--search_mini_size", type=int, default=None)
    args = parser.parse_args()

    if args.single_run:
        Path(args.results_dir).mkdir(parents=True, exist_ok=True)
        metrics = run_experiment(
            program_path=args.program_path,
            results_dir=args.results_dir,
            protocol_mode=args.protocol_mode,
            search_mini_size=args.search_mini_size,
        )
        write_json(Path(args.results_dir) / "metrics.json", metrics)
        print(json.dumps(metrics, indent=2, ensure_ascii=False))
    else:
        main(args.program_path, args.results_dir)


In [ ]:
%%writefile advanced_vqa_task_order_vote_plus/shinka_config.yaml
# ShinkaEvolve 配置：
# - proposal 模型走 8000 上的 coder；
# - embedding 模型走 8002；
# - 本轮不改模型拓扑，只共享一个 8001 VLM 给评测使用；
# - 这里显式覆盖成高预算但稳定的本地搜索设置，不再吃 package 默认 medium_budget。
db:
  num_islands: 5
  archive_size: 40
  num_archive_inspirations: 4
  num_top_k_inspirations: 2
  migration_interval: 10
  migration_rate: 0.1
  parent_selection_strategy: weighted
evo:
  num_generations: 50
  llm_models:
    - "local/cyankiwi/Qwen3-Coder-Next-AWQ-4bit@http://localhost:8000/v1"
  embedding_model: "local/BAAI/bge-m3@http://localhost:8002/v1"
  max_patch_attempts: 3
  max_patch_resamples: 3
  patch_types:
    - diff
    - full
  patch_type_probs:
    - 0.85
    - 0.15
  llm_kwargs:
    temperatures:
      - 0.0
      - 0.4
      - 0.8
    max_tokens: 8192
  use_text_feedback: true


In [ ]:
# 重新确认当前 frozen manifest 已写到磁盘，并打印 actual vs expected 的分布差异。
import json
from create_medframeqa_split_manifest import ensure_split_manifest

manifest_path = ensure_split_manifest(SPLIT_MANIFEST_PATH)
manifest = json.load(open(manifest_path))
print("manifest_path:", manifest_path)
print("manifest_version:", manifest["version"])
print("targets:", manifest["targets"])
print("generator:", manifest["generator"]["strategy"])
print("restart_count:", manifest["generator"].get("restart_count"))
print("objective_score:", manifest.get("objective_score"))
for split_name, stats in manifest["stats"].items():
    print("\n==", split_name, "==")
    print("size:", stats["size"], "groups:", stats["group_count"], "objective:", stats["objective_score"])
    print("actual modality:", stats["modality_distribution"])
    print("expected modality:", stats["expected_modality_distribution"])
    print("modality dev summary:", stats["relative_deviation_summary"]["modality"])
    print("actual answers:", stats["answer_distribution"])
    print("expected answers:", stats["expected_answer_distribution"])
    print("answer dev summary:", stats["relative_deviation_summary"]["answer"])


In [ ]:
# 1-sample sanity check：直接用当前 initial.py 试一题。
import importlib.util
import os
from datasets import load_dataset
from medframeqa_runtime import (
    acquire_vlm_lock,
    get_protocol_subset,
    load_split_manifest,
    make_openai_client,
)

initial_path = os.path.join(TASK_DIR, "initial.py")
spec = importlib.util.spec_from_file_location(f"{TASK_NAME}_initial", initial_path)
initial = importlib.util.module_from_spec(spec)
spec.loader.exec_module(initial)

dataset = load_dataset(
    "SuhaoYu1020/MedFrameQA",
    split="test",
    cache_dir=os.environ.get("MEDFRAMEQA_DATASET_CACHE", "/tmp/medframeqa_hf_cache"),
)
manifest = load_split_manifest(SPLIT_MANIFEST_PATH)
subset, _ = get_protocol_subset(dataset, manifest, "search_mini", 1)
sample = subset[0]
messages = initial.format_vqa_payload(sample)
valid_letters = [chr(65 + index) for index in range(len(sample["options"]))]
client = make_openai_client("http://localhost:8001/v1", timeout=float(DEFAULT_ENV["MEDFRAMEQA_API_TIMEOUT"]))

with acquire_vlm_lock(task_name=TASK_NAME, results_dir=RESULTS_DIR, mode="sanity_check"):
    response = client.chat.completions.create(
        model="cyankiwi/Qwen3.5-35B-A3B-AWQ-4bit",
        messages=messages,
        temperature=0.0,
        max_tokens=4,
        extra_body={
            "guided_choice": valid_letters,
            "continue_final_message": True,
            "add_generation_prompt": False,
        },
    )

print("question_id:", sample["question_id"])
print("raw:", repr(response.choices[0].message.content))
print("gt:", sample["correct_answer"])


In [ ]:
# 5-sample smoke test：确认 evaluator、锁、metrics schema 都能跑通。
import json
import os
import signal
import subprocess
import sys
import time

smoke_results = os.path.join("/tmp", f"advanced_vqa_task_order_vote_plus_smoketest_{RUN_TAG}")
env = dict(os.environ)
env.update(DEFAULT_ENV)
env["MEDFRAMEQA_PROTOCOL_MODE"] = "search_mini"
env["MEDFRAMEQA_SEARCH_MINI_SIZE"] = "5"
evaluate_path = os.path.join(TASK_DIR, "evaluate.py")
initial_path = os.path.join(TASK_DIR, "initial.py")

cmd = [
    sys.executable,
    evaluate_path,
    "--program_path",
    initial_path,
    "--results_dir",
    smoke_results,
    "--single_run",
    "--protocol_mode",
    "search_mini",
    "--search_mini_size",
    "5",
]

proc = subprocess.Popen(cmd, env=env, start_new_session=True)
print("Smoke test PGID:", proc.pid)
try:
    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
except KeyboardInterrupt:
    print("Interrupt received; terminating smoke test process group:", proc.pid)
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    time.sleep(2)
    if proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    raise

print("smoke_results:", smoke_results)
print(json.load(open(os.path.join(smoke_results, "metrics.json"))))


In [ ]:
# 正式运行：阻塞等待 shinka_run 完成，并把输出直接显示在 notebook 里。
import os
import signal
import subprocess
import sys
import time
from pathlib import Path

env = dict(os.environ)
env.update(DEFAULT_ENV)

shinka_bin = Path(sys.executable).with_name("shinka_run")
if not shinka_bin.exists():
    fallback = None
    for prefix in env.get("PATH", "").split(os.pathsep):
        candidate = Path(prefix) / "shinka_run"
        if candidate.exists():
            fallback = candidate
            break
    if fallback is None:
        raise RuntimeError(f"找不到 shinka_run: {shinka_bin}")
    shinka_bin = fallback

cmd = [
    str(shinka_bin),
    "--task-dir",
    TASK_DIR,
    "--config-fname",
    "shinka_config.yaml",
    "--results_dir",
    RESULTS_DIR,
    "--num_generations",
    str(NUM_GENERATIONS),
    "--max-proposal-jobs",
    str(MAX_PROPOSAL_JOBS),
    "--max-evaluation-jobs",
    str(MAX_EVALUATION_JOBS),
]

print("Blocking run command:")
print(" ".join(cmd))
print("RESULTS_DIR:", RESULTS_DIR)

proc = subprocess.Popen(cmd, env=env, cwd=os.path.abspath("."), start_new_session=True)
print("Run PGID:", proc.pid)
try:
    return_code = proc.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)
except KeyboardInterrupt:
    print("Interrupt received; terminating run process group:", proc.pid)
    try:
        os.killpg(proc.pid, signal.SIGTERM)
    except ProcessLookupError:
        pass
    time.sleep(2)
    if proc.poll() is None:
        try:
            os.killpg(proc.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
    raise

print("Run finished. RESULTS_DIR:", RESULTS_DIR)


In [ ]:
# 结果可视化：画每一代 combined_score，并标出 invalid generation。
import math
from medframeqa_runtime import collect_generation_records, select_top_k_records

records = collect_generation_records(RESULTS_DIR)
print("generation_count:", len(records))
if not records:
    raise RuntimeError("当前 RESULTS_DIR 没有 generation 结果。请先完成正式运行。")

top_records = select_top_k_records(records, top_k=3)
print("top3 generations:", [row.get("generation") for row in top_records])

try:
    import matplotlib.pyplot as plt
except ImportError:
    print("matplotlib 不可用，跳过绘图。")
    for row in records:
        print(row.get("generation"), row.get("combined_score"), row.get("invalid_generation"))
else:
    generations = [row.get("generation") for row in records]
    scores = [row.get("combined_score", 0.0) for row in records]
    invalid_generations = [row.get("generation") for row in records if row.get("invalid_generation")]
    invalid_scores = [row.get("combined_score", 0.0) for row in records if row.get("invalid_generation")]

    best_so_far = []
    current_best = -math.inf
    for score in scores:
        current_best = max(current_best, score)
        best_so_far.append(current_best)

    plt.figure(figsize=(10, 4.5))
    plt.plot(generations, scores, color="#1565c0", marker="o", linewidth=1.6, label="combined_score")
    plt.plot(generations, best_so_far, color="#2e7d32", linestyle="--", linewidth=1.4, label="best_so_far")
    if invalid_generations:
        plt.scatter(invalid_generations, invalid_scores, color="#c62828", marker="x", s=80, label="invalid_generation")
    plt.xlabel("Generation")
    plt.ylabel("Combined Score")
    plt.title(TASK_NAME + " evolution curve")
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# 论文汇总：统计 valid/invalid rate，并对 top-3 跑 holdout，最后选一个跑 final test。
import json
import os
import subprocess
import sys

from medframeqa_runtime import (
    collect_generation_records,
    select_best_so_far,
    select_top_k_records,
    write_csv_rows,
    write_json,
)

PAPER_DIR = os.path.join(RESULTS_DIR, "paper_eval")
os.makedirs(PAPER_DIR, exist_ok=True)

def run_direct_eval(program_path, protocol_mode, out_name):
    out_dir = os.path.join(PAPER_DIR, out_name)
    env = dict(os.environ)
    env.update(DEFAULT_ENV)
    cmd = [
        sys.executable,
        os.path.join("advanced_vqa_task_order_vote_plus", "evaluate.py"),
        "--program_path",
        program_path,
        "--results_dir",
        out_dir,
        "--single_run",
        "--protocol_mode",
        protocol_mode,
        "--search_mini_size",
        str(SEARCH_MINI_SIZE),
    ]
    subprocess.run(cmd, env=env, check=True)
    return json.load(open(os.path.join(out_dir, "metrics.json")))

records = collect_generation_records(RESULTS_DIR)
write_csv_rows(os.path.join(PAPER_DIR, "generation_records.csv"), records)
write_json(os.path.join(PAPER_DIR, "generation_records.json"), records)

valid_records = [row for row in records if not row.get("invalid_generation")]
invalid_records = [row for row in records if row.get("invalid_generation")]
valid_rate = (len(valid_records) / len(records)) if records else 0.0
invalid_rate = (len(invalid_records) / len(records)) if records else 0.0
best_record = select_top_k_records(records, top_k=1)
best_record = best_record[0] if best_record else None
last_record = records[-1] if records else None

milestone_rows = []
max_generation = records[-1]["generation"] if records else None
milestone_generations = sorted(
    set(
        [value for value in [10, 20, 30, 40, 49] if isinstance(value, int)]
        + ([max_generation] if max_generation is not None else [])
    )
)
for milestone in milestone_generations:
    record = select_best_so_far(records, milestone)
    if record is None:
        continue
    metrics = run_direct_eval(
        record["program_path"],
        "evolution_pool",
        f"milestone_gen_{milestone}_pool_eval",
    )
    milestone_rows.append({
        "milestone_generation": milestone,
        "selected_generation": record["generation"],
        "selected_program_path": record["program_path"],
        **metrics,
    })
write_csv_rows(os.path.join(PAPER_DIR, "milestone_pool_eval.csv"), milestone_rows)
write_json(os.path.join(PAPER_DIR, "milestone_pool_eval.json"), milestone_rows)

top_records = select_top_k_records(records, top_k=3)
holdout_rows = []
for rank, record in enumerate(top_records, 1):
    metrics = run_direct_eval(
        record["program_path"],
        "selection_holdout",
        f"top{rank}_holdout_eval",
    )
    holdout_rows.append({
        "rank": rank,
        "generation": record["generation"],
        "program_path": record["program_path"],
        **metrics,
    })
write_csv_rows(os.path.join(PAPER_DIR, "top3_holdout_eval.csv"), holdout_rows)
write_json(os.path.join(PAPER_DIR, "top3_holdout_eval.json"), holdout_rows)

best_holdout = None
if holdout_rows:
    best_holdout = sorted(
        holdout_rows,
        key=lambda row: (
            -row.get("holdout_score", 0.0),
            row.get("generation", 10**9),
        ),
    )[0]

summary = {
    "task_name": TASK_NAME,
    "results_dir": RESULTS_DIR,
    "generation_count": len(records),
    "valid_generation_count": len(valid_records),
    "invalid_generation_count": len(invalid_records),
    "valid_generation_rate": round(valid_rate, 4),
    "invalid_generation_rate": round(invalid_rate, 4),
    "invalid_rate_threshold": INVALID_RATE_THRESHOLD,
    "paper_ready_candidate": invalid_rate <= INVALID_RATE_THRESHOLD,
    "best_generation": best_record["generation"] if best_record else None,
    "best_combined_score": best_record.get("combined_score") if best_record else None,
    "last_generation": last_record["generation"] if last_record else None,
    "last_combined_score": last_record.get("combined_score") if last_record else None,
    "top3_generations": [row["generation"] for row in top_records],
    "milestone_pool": milestone_rows,
    "top3_holdout": holdout_rows,
}
if best_holdout is not None:
    final_metrics = run_direct_eval(
        best_holdout["program_path"],
        "independent_final_test",
        "final_test_eval",
    )
    summary["selected_generation"] = best_holdout["generation"]
    summary["selected_program_path"] = best_holdout["program_path"]
    summary["final_test"] = final_metrics

write_json(os.path.join(PAPER_DIR, "paper_summary.json"), summary)
print(json.dumps(summary, indent=2, ensure_ascii=False))
